<a href="https://colab.research.google.com/github/AdityaKrNag007/House-Cost-Prediction-ML/blob/main/house_predict_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [191]:
import pandas as pd
import numpy as np

data = pd.read_csv('/content/train_houses.csv')
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [192]:
features = ['LotArea','YearBuilt','1stFlrSF','2ndFlrSF',
            'FullBath','BedroomAbvGr','TotRmsAbvGrd',
            'GarageArea','GrLivArea']
features += ['OverallQual','GarageCars']

X = data[features]
X = X.dropna(axis=0)

X['HouseAge'] = 2024 - X['YearBuilt']
X['TotalSF'] = X['1stFlrSF'] + X['2ndFlrSF']
X['TotalRooms'] = X['TotRmsAbvGrd'] + X['BedroomAbvGr']

y = data['SalePrice']

In [193]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, random_state=1)

In [194]:
X_train["Qual_GrLiv"] = X_train["OverallQual"] * X_train["GrLivArea"]
X_val["Qual_GrLiv"] = X_val["OverallQual"] * X_val["GrLivArea"]
X_test["Qual_GrLiv"] = X_test["OverallQual"] * X_test["GrLivArea"]

In [195]:
from sklearn.ensemble import RandomForestRegressor

model_RF = RandomForestRegressor(
    n_estimators=400,
    max_depth=15,
    min_samples_split=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1                              )

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

model_RF.fit(X_train,y_train_log)

RandomForestRegressor(max_depth=15, min_samples_leaf=2, min_samples_split=8,
                      n_estimators=400, n_jobs=-1, random_state=42)

In [196]:
preds_RF_val = model_RF.predict(X_val)

In [197]:
from sklearn.metrics import mean_absolute_error

mae_RF_val = mean_absolute_error(y_val, preds_RF_val)
print("mae_RF_val : ",mae_RF_val)

mae_RF_val :  179407.1865222533


In [198]:
preds_RF_test = model_RF.predict(X_test)
mae_RF_test = mean_absolute_error(y_test, preds_RF_test)

In [199]:
preds_RF_train = model_RF.predict(X_train)

mae_RF_train = mean_absolute_error(y_train,preds_RF_train)

In [200]:
print("mae_RF_train : ",mae_RF_train)
print("mae_RF_val : ",mae_RF_val)
print("mae_RF_test",mae_RF_test)

mae_RF_train :  183126.01925681424
mae_RF_val :  179407.1865222533
mae_RF_test 175760.6212150447


In [201]:
importances = model_RF.feature_importances_
features_check = X_train.columns

feat_df = pd.DataFrame({
    "Feature": features_check,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feat_df

,Feature,Importance
14,Qual_GrLiv,0.712264
2,1stFlrSF,0.043497
9,OverallQual,0.042944
1,YearBuilt,0.036482
11,HouseAge,0.035457
7,GarageArea,0.034757
0,LotArea,0.032500
10,GarageCars,0.020397
3,2ndFlrSF,0.013399
12,TotalSF,0.010215
